# Run MeanFlow on Google Colab GPU

Before running the cells: in Colab choose **Runtime > Change runtime type > GPU**.

The setup cell below clones (or pulls) this GitHub repo directly into the Colab VM, so any changes you push from VS Code are picked up by rerunning that cell. If the repo can't be cloned for some reason, it falls back to a manual zip upload.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/nonplayercharacter3/MeanFlow-one-step-pixel-generation.git"
REPO_DIR = Path("/content/MeanFlow-one-step-pixel-generation")

if REPO_DIR.exists():
    print("Repo already present, pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

project_root = REPO_DIR
%cd {project_root}

if not (project_root / "train.py").exists():
    print("train.py was not found after cloning.")
    print("Falling back to manual upload: zip your whole MeanFlow folder locally and upload it now.")
    from google.colab import files
    import zipfile

    uploaded = files.upload()
    zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_files:
        raise RuntimeError("Please upload a .zip file containing train.py, meanflow.py, model.py, utils.py, and data/.")

    zip_path = Path(zip_files[0])
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content")

    matches = list(Path("/content").rglob("train.py"))
    if not matches:
        raise RuntimeError("Could not find train.py after unzipping.")

    project_root = matches[0].parent
    %cd {project_root}
else:
    print(f"Using project folder: {project_root}")

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -r requirements.txt

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Colab, set Runtime > Change runtime type > GPU, then rerun.")

CUDA available: True
GPU: Tesla T4


## Short GPU smoke test

This uses the included `data/smoke_test.png` and writes outputs to `outputs/colab_smoke`.

In [ ]:
!python train.py \
  --images data/smoke_test.png \
  --steps 100 \
  --batch-size 8 \
  --sample-every 50 \
  --checkpoint-every 100 \
  --output-dir outputs/colab_smoke

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for path in sorted(Path("outputs/colab_smoke").glob("sample_step_*.png")):
    print(path)
    display(Image(filename=str(path)))

## Longer one-image run

Run this after the smoke test works. Increase `--steps` if you want a longer overfit experiment.

In [ ]:
!python train.py \
  --images data/overfit1/imagenette_one.png \
  --steps 5000 \
  --batch-size 8 \
  --lr 3e-4 \
  --hidden-channels 256 \
  --num-blocks 6 \
  --sample-every 500 \
  --checkpoint-every 1000 \
  --output-dir outputs/imagenette_one_overfit

## Three-image overfit run (required assignment milestone)

Trains on `data/overfit3/image_0.png`, `image_1.png`, `image_2.png`. Saves per-image cleans/samples (`clean_0.png`, `sample_best_0.png`, ...) plus `clean_grid.png` / `sample_best_grid.png` contact sheets.

In [ ]:
!python train.py \
  --images data/overfit3/image_0.png data/overfit3/image_1.png data/overfit3/image_2.png \
  --steps 5000 \
  --batch-size 9 \
  --lr 3e-4 \
  --hidden-channels 256 \
  --num-blocks 6 \
  --sample-every 500 \
  --checkpoint-every 1000 \
  --output-dir outputs/overfit3

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

output_dir = "outputs/overfit3"

clean = Image.open(f"{output_dir}/clean_grid.png")
best = Image.open(f"{output_dir}/sample_best_grid.png")
history = pd.read_csv(f"{output_dir}/loss_history.csv")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(clean)
axes[0].set_title("Clean (target)")
axes[0].axis("off")
axes[1].imshow(best)
axes[1].set_title("Best one-step samples")
axes[1].axis("off")
axes[2].plot(history["step"], history["loss"], label="loss")
axes[2].plot(history["step"], history["sample_mse"], label="sample_mse")
axes[2].set_xlabel("step")
axes[2].set_yscale("log")
axes[2].legend()
axes[2].set_title("Training curves")
plt.tight_layout()
plt.show()

ckpt = torch.load(f"{output_dir}/checkpoint_best.pt", map_location="cpu")
print("Best step:", ckpt["step"], "sample_mse:", ckpt["sample_mse"])

## Save outputs to Google Drive

Colab VMs are ephemeral — everything in `outputs/` is lost if the runtime disconnects or the session ends. Run this cell after each experiment to copy results to Drive so they survive across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

drive_dir = Path('/content/drive/MyDrive/MeanFlow_outputs')
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree('outputs', drive_dir, dirs_exist_ok=True)
print(f"Copied outputs/ to {drive_dir}")